# Detector de Juguetes: Transfer Learning + Fine-Tuning con YOLOv8

**Clases:** `luchador`, `rescatista` · **Dataset:** Roboflow `detector-juguetes` v1

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.

Ejecuta las celdas en orden con el botón ▶ (o Shift + Enter).

## 1. Verificar la GPU
Debe aparecer una tabla con **Tesla T4**. Si da error, activa la GPU (ver arriba).

In [ ]:
!nvidia-smi

## 2. Instalar Ultralytics (YOLO) y Roboflow

In [ ]:
!pip install -q -U ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Descargar el dataset desde Roboflow

**Borra el contenido de esta celda y pega el código que te da Roboflow** en:
*Versions → v1-tablet-iphone → Download Dataset → formato **YOLOv8** → Show download code → Jupyter*.

⚠️ Ese código contiene tu **API key**: no compartas el notebook con ella.

In [ ]:
# Codigo de Roboflow (v1-tablet-iphone, formato YOLOv8). La API key se pide al ejecutar: nunca la escribas aqui.
from roboflow import Roboflow
from getpass import getpass
rf = Roboflow(api_key=getpass("API key de Roboflow: "))  # se pide al ejecutar y no queda guardada
project = rf.workspace("nobodys-workspace-uvclt").project("detector-juguetes")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
# Revisar que el dataset llegó bien: clases y cantidad de imágenes por conjunto
import os, yaml
DATA = f"{dataset.location}/data.yaml"
print(yaml.safe_load(open(DATA)))
for s in ['train', 'valid', 'test']:
    p = f"{dataset.location}/{s}/images"
    print(s, len(os.listdir(p)) if os.path.exists(p) else 0, 'imágenes')

## 4. Entrenar: Transfer Learning + Fine-Tuning

- `yolov8n.pt` = modelo **preentrenado con COCO** (80 clases). Ya sabe ver formas y bordes → **transfer learning**.
- `model.train(...)` ajusta sus pesos con nuestras imágenes → **fine-tuning**.
- `epochs=80`: veces que recorre todo el dataset. `patience=20`: se detiene solo si deja de mejorar.
- YOLO aplica *augmentation* automáticamente (mosaic, flips, brillo/color, escala).

En una T4 tarda unos minutos.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # modelo preentrenado (transfer learning)
results = model.train(
    data=DATA,
    epochs=80,
    patience=20,
    imgsz=640,
    batch=16,
    name='juguetes_v1',
)

## 5. Resultados (para la exposición)

- **mAP50**: qué tan bien detecta (0 a 1). Más de **0.8** es muy bueno.
- **Matriz de confusión**: si confunde luchador con rescatista.
- **results.png**: curvas de pérdida (*loss*), que deben bajar, y mAP, que debe subir.

In [ ]:
from IPython.display import Image, display
RUN = model.trainer.save_dir   # carpeta del entrenamiento
print('Carpeta:', RUN)
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'{RUN}/{f}'
    if os.path.exists(p):
        print(f); display(Image(filename=p, width=900))

## 6. Evaluar con el conjunto de TEST (imágenes que el modelo nunca vio)

In [ ]:
best = YOLO(f'{RUN}/weights/best.pt')
m = best.val(data=DATA, split='test')
print(f'mAP50 (test): {m.box.map50:.3f}   mAP50-95 (test): {m.box.map:.3f}')

In [ ]:
# Ver predicciones sobre algunas imágenes de test
import glob
imgs = sorted(glob.glob(f'{dataset.location}/test/images/*'))[:6]
for r in best.predict(imgs, conf=0.5, save=True):
    pass
for p in sorted(glob.glob(f'{r.save_dir}/*'))[:6]:
    display(Image(filename=p, width=320))

## 7. Exportar para la tablet (LiteRT / TFLite)

`nms=None` = el filtro de recuadros duplicados (*Non-Maximum Suppression*) lo hace la app en la tablet, como pide el plugin `ultralytics_yolo`.

In [ ]:
best.export(format='litert', nms=None, imgsz=640)
tfl = [p for p in glob.glob(f'{RUN}/weights/**/*.tflite', recursive=True)]
print(tfl)

## 8. Descargar todo en un ZIP
Contiene `best.pt` (el modelo fine-tuneado), `juguetes.tflite` (para la app) y las gráficas.

In [ ]:
import shutil
OUT = '/content/modelo_juguetes_v1'
os.makedirs(OUT, exist_ok=True)
shutil.copy(f'{RUN}/weights/best.pt', f'{OUT}/best.pt')
# preferimos la versión float32 si hay varias
tfl_sorted = sorted(tfl, key=lambda p: ('float32' not in p, p))
shutil.copy(tfl_sorted[0], f'{OUT}/juguetes.tflite')
for f in ['results.png', 'confusion_matrix.png', 'results.csv']:
    if os.path.exists(f'{RUN}/{f}'): shutil.copy(f'{RUN}/{f}', OUT)
shutil.make_archive(OUT, 'zip', OUT)
from google.colab import files
files.download(OUT + '.zip')